# LoRA Lesson

Notebook version of `lora-lesson.py`, organized into small runnable cells.

In [1]:
import re
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"


In [3]:
def make_dataset():
    """Build a small arithmetic dataset with strict output-format instructions."""
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            rows.append({
                "prompt": f"What is {a} + {b}? Respond exactly as <think>...</think><answer>...</answer>",
                "answer": str(a + b),
            })

    return Dataset.from_list(rows)


In [4]:
dataset = make_dataset()
split = dataset.train_test_split(test_size=0.25, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]


In [5]:
def extract_answer(text):
    """Return the contents of the first <answer> tag, or an empty string."""
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else ""


In [6]:
def has_required_format(text):
    """Check whether the response contains both think and answer tags."""
    return bool(re.search(
        r"<think>.*?</think>\s*<answer>.*?</answer>",
        text,
        re.DOTALL
    ))


In [7]:
def format_reward(completions, **kwargs):
    """Reward completions that follow the required XML-like response format."""
    rewards = []

    for c in completions:
        text = c[0]["content"] if isinstance(c, list) else c
        rewards.append(0.5 if has_required_format(text) else 0.0)

    return rewards


In [8]:
def correctness_reward(completions, answer, **kwargs):
    """Reward completions whose extracted answer matches the expected answer."""
    rewards = []

    for c, expected in zip(completions, answer):
        text = c[0]["content"] if isinstance(c, list) else c
        predicted = extract_answer(text)
        rewards.append(1.0 if predicted == expected else 0.0)

    return rewards


In [9]:
def generate_response(model, tokenizer, prompt):
    """Generate a deterministic response for a single user prompt."""
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


In [10]:
def evaluate_model(model, tokenizer, eval_dataset, label):
    """Run evaluation on a dataset and print summary metrics with examples."""
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    total_reward = 0.0

    examples = []

    for row in eval_dataset:
        prompt = row["prompt"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        predicted = extract_answer(text)

        is_formatted = has_required_format(text)
        is_correct = predicted == expected

        format_score = 0.5 if is_formatted else 0.0
        correctness_score = 1.0 if is_correct else 0.0
        reward = format_score + correctness_score

        formatted += int(is_formatted)
        correct += int(is_correct)
        total_reward += reward

        if len(examples) < 5:
            examples.append({
                "prompt": prompt,
                "expected": expected,
                "generated": text,
                "predicted": predicted,
                "reward": reward,
            })

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {correct / total:.2%}")
    print(f"Format compliance: {formatted}/{total} = {formatted / total:.2%}")
    print(f"Average reward:    {total_reward / total:.3f}")

    print("\nSample generations:")
    for ex in examples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Reward:   ", ex["reward"])


In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 228.73it/s]


In [13]:
evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before GRPO + LoRA"
)



=== Before GRPO + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Average reward:    0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3 = 19</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7 = 10</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6 = 18</think>
Predicted: 
Reward:    0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  12
Generated: <think>5 + 7 = 12</

In [14]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [15]:
training_args = GRPOConfig(
    output_dir="grpo-arithmetic-lora-demo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=64,
    num_train_epochs=1,
    logging_steps=10,
    learning_rate=5e-5,
)


In [16]:
trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=[format_reward, correctness_reward],
    peft_config=lora_config,
)


In [17]:
trainer.train()
trainer.save_model("grpo-arithmetic-lora-adapter")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,-0.000000
20,-0.000000
30,-0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000


In [18]:
trained_model = trainer.model


In [19]:
evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After GRPO + LoRA"
)



=== After GRPO + LoRA ===
Answer accuracy:   49/50 = 98.00%
Format compliance: 50/50 = 100.00%
Average reward:    1.480

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>...</think><answer>...</answer>
Expected:  19
Generated: <think>16</think>
<answer>20</answer>
Predicted: 20
Reward:    0.5
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>...</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7 = 10</think><answer>10</answer>
Predicted: 10
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>...</think><answer>...</answer>
Expected:  18
Generated: <think>12</think>
<answer>18</answer>
Predicted: 18
Reward:    1.5
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <think>...</think><answer>...</a